# L5: Cross-Device Memory Orchestration

A single EdgeShard on one device is useful, but the real power comes from **orchestrating memory across devices**. Your glasses capture meeting discussions and whiteboard diagrams, your phone stores calendar events and emails, and a cloud server provides the full searchable history.

In this lesson, you'll build the meeting use case from the Qdrant Context Hub:

```
Glasses (meeting room)         Phone              Cloud
+--------------------+    +----------+    +-----------+
| Mutable Shard      |    | Calendar |    | Qdrant    |
| - whiteboard notes |--->| - emails |--->| Server    |
| - discussion points|    | - notes  |    | (history) |
| - action items     |    +----------+    +-----------+
+--------------------+         |               |
         |                     v               v
         +------- Cascade Query: "What were the action items?" ------+
```

You'll learn to:
- Set up a dual-shard architecture (mutable + immutable)
- Implement cascade queries: search local first, then cloud
- Sync edge data to a Qdrant server
- Initialize an edge shard from a server snapshot
- Compile models for multiple Snapdragon devices via AI Hub

## Setup

In [ ]:
!pip install qdrant-edge-py qdrant-client qai-hub "qai-hub-models[nomic_embed_text]" torch transformers numpy

In [ ]:
import os
import sys
sys.path.append("..")

import torch
import numpy as np
import time
from pathlib import Path
from collections import deque
from transformers import AutoTokenizer
from qai_hub_models.models.nomic_embed_text import Model as NomicEmbedText
from qdrant_edge import (
    EdgeShard, EdgeConfig, VectorDataConfig, Distance,
    Point, UpdateOperation, Query, QueryRequest,
)
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, PointStruct

# Load embedding model
text_model = NomicEmbedText.from_pretrained()
tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")

EMBEDDING_DIM = 512
MAX_SEQ_LEN = 512
VECTOR_NAME = "memory"

def embed(texts, prefix="search_document: "):
    """Generate embeddings using the nomic model."""
    prefixed = [prefix + t for t in texts]
    encoded = tokenizer(
        prefixed, padding="max_length", truncation=True,
        max_length=MAX_SEQ_LEN, return_tensors="pt",
    )
    with torch.no_grad():
        return text_model(encoded["input_ids"], encoded["attention_mask"])

print(f"Embedding model loaded: nomic_embed_text ({EMBEDDING_DIM}d)")

## 1. Compile for Multiple Snapdragon Devices

In a multi-device setup, each device may have a different Snapdragon chipset. AI Hub lets you compile the same model for different targets.

```
AR Glasses (Snapdragon XR2)  --> compiled_glasses.tflite
Phone (Snapdragon 8 Gen 3)   --> compiled_phone.tflite
Tablet (Snapdragon 8 Gen 2)  --> compiled_tablet.tflite
```

In [ ]:
import qai_hub
from utils import get_ai_hub_api_token

ai_hub_api_token = get_ai_hub_api_token()
!qai-hub configure --api_token $ai_hub_api_token

# Trace the model once
example_ids = torch.randint(0, tokenizer.vocab_size, (1, MAX_SEQ_LEN))
example_mask = torch.ones(1, MAX_SEQ_LEN, dtype=torch.long)
traced_model = torch.jit.trace(text_model, (example_ids, example_mask))

# Compile for multiple target devices
target_devices = [
    "Samsung Galaxy S23",
    "Samsung Galaxy S24",
    "Samsung Galaxy Tab S8",
]

compiled_models = {}
for dev_name in target_devices:
    print(f"Compiling for {dev_name}...")
    dev = qai_hub.Device(dev_name)
    job = qai_hub.submit_compile_job(
        model=traced_model,
        input_specs={"input_ids": (1, MAX_SEQ_LEN), "attention_mask": (1, MAX_SEQ_LEN)},
        device=dev,
    )
    compiled_models[dev_name] = job.get_target_model()
    print(f"  Done: {job.job_id}")

print(f"\nCompiled for {len(compiled_models)} devices")

## 2. The Dual-Shard Architecture

Production Qdrant Edge deployments use two shards on each device:

- **Mutable shard**: Stores new data captured locally. Unindexed for fast writes.
- **Immutable shard**: Mirrors the server collection via snapshots. HNSW-indexed for fast search.

Queries run against both shards, and results are merged.

```
Device                          Cloud
+-----------------+             +-----------+
| Mutable Shard   | --sync-->   | Qdrant    |
| (new local data)|             | Server    |
+-----------------+             +-----------+
| Immutable Shard | <--snapshot |           |
| (server mirror) |             |           |
+-----------------+             +-----------+
```

In [ ]:
# Create the mutable shard (for new local data)
MUTABLE_DIR = "./mutable_shard"
Path(MUTABLE_DIR).mkdir(parents=True, exist_ok=True)

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=EMBEDDING_DIM,
            distance=Distance.Cosine,
        )
    }
)

mutable_shard = EdgeShard(MUTABLE_DIR, config)
print("Mutable shard ready (for local writes)")

## 3. Set Up a Cloud Server with Meeting History

We'll use `qdrant-client` in memory mode to simulate a cloud Qdrant server. In production, this would be a real Qdrant Cloud instance.

The cloud holds weeks of historical meeting data: past decisions, action items, and discussion summaries from previous meetings.

In [ ]:
cloud_client = QdrantClient(":memory:")

COLLECTION_NAME = "device_memories"

cloud_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance="Cosine",
    )
)

# Historical meeting data spanning the past 3 weeks
# This simulates what a team accumulates over time
historical_meetings = [
    # Week 1: Sprint planning and architecture
    {"text": "Sprint planning: committed to 34 story points, focus on edge sync module", "meeting": "sprint-planning", "week": 1},
    {"text": "Architecture review: proposed dual-shard pattern for edge devices", "meeting": "arch-review", "week": 1},
    {"text": "Architecture review: Sarah suggested using HNSW indexing for immutable shards", "meeting": "arch-review", "week": 1},
    {"text": "Architecture review: decided cascade queries search local first then cloud", "meeting": "arch-review", "week": 1},
    {"text": "Standup: John blocked on Snapdragon compile pipeline, needs support", "meeting": "standup", "week": 1},
    {"text": "Standup: Maria finished the sync queue implementation", "meeting": "standup", "week": 1},
    {"text": "1:1 with manager: discussed promotion timeline and project ownership", "meeting": "one-on-one", "week": 1},
    {"text": "Design review: whiteboard diagram of cross-device data flow", "meeting": "design-review", "week": 1},
    {"text": "Design review: action item for John to benchmark latency on Galaxy S24", "meeting": "design-review", "week": 1},
    {"text": "Design review: agreed on 20ms P95 latency target for edge queries", "meeting": "design-review", "week": 1},
    {"text": "Retro: team velocity improved 15% after switching to smaller PRs", "meeting": "retro", "week": 1},
    {"text": "Retro: action item to add integration tests for sync module", "meeting": "retro", "week": 1},

    # Week 2: Implementation and partnerships
    {"text": "Sprint planning: 28 points committed, carrying over sync bug fixes", "meeting": "sprint-planning", "week": 2},
    {"text": "Standup: edge shard persistence tests passing on all three devices", "meeting": "standup", "week": 2},
    {"text": "Standup: Sarah started profiling memory usage on Galaxy Tab S8", "meeting": "standup", "week": 2},
    {"text": "Partnership meeting: discussed AI Hub integration for course", "meeting": "partnership", "week": 2},
    {"text": "Partnership meeting: offered access to RB3 robotics dev kit", "meeting": "partnership", "week": 2},
    {"text": "Partnership meeting: agreed to demo at Mobile World Congress", "meeting": "partnership", "week": 2},
    {"text": "Tech talk: Maria presented on vector quantization for edge deployment", "meeting": "tech-talk", "week": 2},
    {"text": "Design review: phone shard should store calendar and email context", "meeting": "design-review", "week": 2},
    {"text": "Design review: glasses capture rate set to 1 observation per 30 seconds", "meeting": "design-review", "week": 2},
    {"text": "Retro: sync queue had a race condition, added mutex lock", "meeting": "retro", "week": 2},
    {"text": "1:1 with manager: green light to present context hub at developer day", "meeting": "one-on-one", "week": 2},

    # Week 3: Testing and optimization
    {"text": "Sprint planning: final sprint before demo, 22 points, all bug fixes", "meeting": "sprint-planning", "week": 3},
    {"text": "Standup: latency benchmarks show 8ms mean on Galaxy S24, well under target", "meeting": "standup", "week": 3},
    {"text": "Standup: John found battery drain issue with continuous glasses capture", "meeting": "standup", "week": 3},
    {"text": "Standup: reduced capture rate to 1 per minute, battery impact now under 12%", "meeting": "standup", "week": 3},
    {"text": "Bug triage: duplicate points appearing after sync, need dedup by ID", "meeting": "bug-triage", "week": 3},
    {"text": "Bug triage: immutable shard not loading after device restart, path issue", "meeting": "bug-triage", "week": 3},
    {"text": "Team sync: Krishna confirmed nomic_embed_text compiles cleanly for XR2 Gen 2", "meeting": "team-sync", "week": 3},
    {"text": "Team sync: CLIP visual encoder needs input quantization for glasses chipset", "meeting": "team-sync", "week": 3},
    {"text": "Demo prep: end-to-end flow working, glasses to phone to cloud", "meeting": "demo-prep", "week": 3},
    {"text": "Demo prep: action item to record backup video in case of live demo failure", "meeting": "demo-prep", "week": 3},
    {"text": "Demo prep: Sarah will present the architecture, John handles live coding", "meeting": "demo-prep", "week": 3},
    {"text": "Retro: cascade query fallback saved us when cloud was briefly unreachable", "meeting": "retro", "week": 3},
    {"text": "Retro: action item to add offline mode indicator in the UI", "meeting": "retro", "week": 3},
]

# Embed all historical data
hist_texts = [m["text"] for m in historical_meetings]
hist_embeddings = embed(hist_texts)

now = time.time()
cloud_client.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        PointStruct(
            id=i + 1000,
            vector=emb.tolist(),
            payload={
                "text": m["text"],
                "source": "cloud",
                "meeting_type": m["meeting"],
                "week": m["week"],
                "timestamp": now - (4 - m["week"]) * 7 * 86400 + i * 3600,
            }
        )
        for i, (m, emb) in enumerate(zip(historical_meetings, hist_embeddings))
    ]
)

print(f"Cloud server ready with {len(historical_meetings)} historical meeting records")
print(f"Spanning {len(set(m['meeting'] for m in historical_meetings))} meeting types over 3 weeks")

## 4. Glasses Capture: Live Meeting Observations

This simulates what happens during a real meeting. The glasses continuously observe the room, capturing whiteboard content, speaker contributions, and action items. Each observation is written to the local mutable shard and queued for cloud sync.

This maps to the Context Hub meeting use case: glasses capture, features extracted on-device, stored locally, then synced.

In [ ]:
sync_queue = deque()

def capture_memory(point_id, text, location="unknown", device="glasses", obs_type="general"):
    """Capture a new memory: store locally and queue for cloud sync."""
    embedding = embed([text])
    emb_list = embedding[0].tolist()
    ts = time.time()

    # Write to local mutable shard
    point = Point(
        id=point_id,
        vector={VECTOR_NAME: emb_list},
        payload={
            "text": text,
            "location": location,
            "device": device,
            "observation_type": obs_type,
            "timestamp": ts,
            "synced": False,
        }
    )
    mutable_shard.update(UpdateOperation.upsert_points([point]))

    # Queue for server sync
    sync_queue.append(PointStruct(
        id=point_id,
        vector=emb_list,
        payload={
            "text": text,
            "location": location,
            "device": device,
            "observation_type": obs_type,
            "timestamp": ts,
        }
    ))

    return point

# Simulate a 45-minute meeting captured by AR glasses
# Glasses observe whiteboard, speakers, and discussions in real time
meeting_observations = [
    # Opening (minute 0-5)
    (0, "Meeting started: Q4 planning review with Sarah, John, Maria, and Krishna", "whiteboard"),
    (1, "Whiteboard: title slide says Q4 Edge Deployment Roadmap", "whiteboard"),
    (2, "Sarah opens: reviewing progress on the context hub since last quarter", "discussion"),
    (3, "Sarah: we hit 8ms mean latency on Galaxy S24, beating our 20ms target", "discussion"),

    # Architecture discussion (minute 5-15)
    (4, "Whiteboard: diagram showing glasses, phone, and cloud with arrows between them", "whiteboard"),
    (5, "John: the dual-shard pattern is working well but we need better dedup", "discussion"),
    (6, "Whiteboard: John draws sync flow with mutex locks at each stage", "whiteboard"),
    (7, "Maria: we should add a version vector to each point for conflict resolution", "discussion"),
    (8, "Sarah: agrees, version vectors prevent the duplicate bug we saw last sprint", "discussion"),
    (9, "Whiteboard: updated architecture now shows version vectors on sync arrows", "whiteboard"),

    # Product comparison demo (minute 15-25)
    (10, "Krishna: demoing the product comparison feature on a Galaxy S23", "discussion"),
    (11, "Krishna: user photographs headphones at Best Buy, price shows $299", "discussion"),
    (12, "Krishna: later at Target, same headphones, CLIP finds the match at 0.94 similarity", "discussion"),
    (13, "Krishna: cascade query pulls the Best Buy price from cloud in 45ms total", "discussion"),
    (14, "Whiteboard: Krishna writes latency breakdown: 8ms local, 37ms cloud roundtrip", "whiteboard"),

    # Robotics discussion (minute 25-35)
    (15, "Sarah: we received the RB3 dev kit for the robotics use case", "discussion"),
    (16, "John: nomic_embed_text compiled for RB3, runs at 12ms per embedding", "discussion"),
    (17, "Whiteboard: robot navigation diagram with spatial memory grid", "whiteboard"),
    (18, "Maria: robot memory decay is important, we can't store everything forever on 4GB RAM", "discussion"),
    (19, "Maria: propose keeping last 1000 observations, evicting by timestamp", "discussion"),

    # Action items and wrap-up (minute 35-45)
    (20, "Action item: John to implement version vectors in the sync module by Friday", "action-item"),
    (21, "Action item: Maria to add memory decay with 1000-point cap to robot agent", "action-item"),
    (22, "Action item: Krishna to record product comparison demo video for MWC", "action-item"),
    (23, "Action item: Sarah to update the architecture doc with version vector changes", "action-item"),
    (24, "Action item: schedule follow-up meeting next Tuesday to review progress", "action-item"),
    (25, "Meeting ended: next sync is Tuesday at 2pm", "discussion"),
]

print("Glasses capturing meeting observations...")
for pid, text, obs_type in meeting_observations:
    capture_memory(pid, text, location="conference-room", device="glasses", obs_type=obs_type)

print(f"Captured {len(meeting_observations)} observations from the meeting")
print(f"  Whiteboard captures: {sum(1 for _, _, t in meeting_observations if t == 'whiteboard')}")
print(f"  Discussion points:   {sum(1 for _, _, t in meeting_observations if t == 'discussion')}")
print(f"  Action items:        {sum(1 for _, _, t in meeting_observations if t == 'action-item')}")
print(f"Sync queue depth: {len(sync_queue)}")

## 5. Cascade Query: Local First, Cloud Fallback

The cascade query pattern searches the local glasses shard first. If results aren't confident enough or the user needs historical context, it falls back to the cloud.

This is what happens after the meeting when someone asks: "What did we decide about latency targets?" The glasses have today's discussion. The cloud has last week's architecture review where the target was originally set.

In [ ]:
def cascade_query(query_text, local_limit=3, cloud_limit=3, score_threshold=0.5):
    """Search local first, fall back to cloud if needed."""
    query_emb = embed([query_text], prefix="search_query: ")
    query_list = query_emb[0].tolist()

    # Step 1: Search local mutable shard (glasses)
    local_results = mutable_shard.query(
        QueryRequest(
            query=Query.Nearest(query_list, using=VECTOR_NAME),
            limit=local_limit,
            with_vector=False,
            with_payload=True,
        )
    )

    # Step 2: Check if local results are sufficient
    good_local = [r for r in local_results if r.score >= score_threshold]

    # Step 3: Fall back to cloud if needed
    cloud_results = []
    if len(good_local) < local_limit:
        cloud_hits = cloud_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_list,
            limit=cloud_limit,
        ).points
        cloud_results = cloud_hits

    return local_results, cloud_results

# Query 1: Today's action items (found locally on glasses)
print("Query: 'What are the action items from today?'")
local, cloud = cascade_query("action items assigned to team members")
print(f"  Local (glasses): {len(local)} results")
for r in local:
    print(f"    [{r.score:.3f}] {r.payload['text']}")
if cloud:
    print(f"  Cloud: {len(cloud)} results")
    for r in cloud:
        print(f"    [{r.score:.3f}] {r.payload.get('text', 'N/A')}")
else:
    print("  Cloud: not needed (local results sufficient)")

print()

# Query 2: Historical context (needs cloud fallback)
print("Query: 'What was decided about the partnership and demo plans?'")
local, cloud = cascade_query("partnership agreement and demo plans")
print(f"  Local (glasses): {len(local)} results")
for r in local:
    print(f"    [{r.score:.3f}] {r.payload['text']}")
print(f"  Cloud (history): {len(cloud)} results")
for r in cloud:
    print(f"    [{r.score:.3f}] {r.payload.get('text', 'N/A')}")

print()

# Query 3: Whiteboard content (found locally)
print("Query: 'What was drawn on the whiteboard?'")
local, cloud = cascade_query("whiteboard diagrams and drawings")
print(f"  Local (glasses): {len(local)} results")
for r in local:
    obs = r.payload.get('observation_type', 'unknown')
    print(f"    [{r.score:.3f}] [{obs}] {r.payload['text']}")

print()

# Query 4: Cross-session question (needs both)
print("Query: 'What latency numbers have we measured?'")
local, cloud = cascade_query("latency measurements and benchmarks")
print(f"  Local (glasses): {len(local)} results")
for r in local:
    print(f"    [{r.score:.3f}] {r.payload['text']}")
print(f"  Cloud (history): {len(cloud)} results")
for r in cloud:
    print(f"    [{r.score:.3f}] {r.payload.get('text', 'N/A')}")

## 6. Sync Edge Data to Cloud

When connectivity is available, flush the sync queue to the cloud server.

In [ ]:
def sync_to_cloud(batch_size=10):
    """Push queued points from edge to cloud."""
    synced = 0
    batch = []

    while sync_queue:
        batch.append(sync_queue.popleft())
        if len(batch) >= batch_size:
            cloud_client.upsert(
                collection_name=COLLECTION_NAME,
                points=batch,
            )
            synced += len(batch)
            batch = []

    if batch:
        cloud_client.upsert(
            collection_name=COLLECTION_NAME,
            points=batch,
        )
        synced += len(batch)

    return synced

print(f"Queue depth before sync: {len(sync_queue)}")
count = sync_to_cloud()
print(f"Synced {count} points to cloud")
print(f"Queue depth after sync: {len(sync_queue)}")

info = cloud_client.get_collection(COLLECTION_NAME)
print(f"Cloud collection now has {info.points_count} points")

## 7. Initialize Edge from Server Snapshot

When setting up a new device (or after a reset), you can initialize the immutable shard from a server snapshot. This gives the device access to the full history immediately.

In production, you'd download a partial snapshot via HTTP. Here we simulate the pattern.

In [ ]:
IMMUTABLE_DIR = "./immutable_shard"
Path(IMMUTABLE_DIR).mkdir(parents=True, exist_ok=True)

immutable_shard = EdgeShard(IMMUTABLE_DIR, config)

# Fetch all points from cloud and populate immutable shard
all_cloud_points = cloud_client.scroll(
    collection_name=COLLECTION_NAME,
    limit=100,
    with_payload=True,
    with_vectors=True,
)[0]

edge_points = [
    Point(
        id=p.id if isinstance(p.id, int) else hash(p.id) % (2**31),
        vector={VECTOR_NAME: p.vector},
        payload=dict(p.payload),
    )
    for p in all_cloud_points
]

immutable_shard.update(UpdateOperation.upsert_points(edge_points))
print(f"Immutable shard initialized with {len(edge_points)} points from cloud")

## 8. Unified Query Across Both Shards

Search both the mutable and immutable shards, merge results, and deduplicate by point ID.

In [ ]:
def unified_query(query_text, limit=5):
    """Query both shards and merge results."""
    query_emb = embed([query_text], prefix="search_query: ")[0].tolist()
    request = QueryRequest(
        query=Query.Nearest(query_emb, using=VECTOR_NAME),
        limit=limit,
        with_vector=False,
        with_payload=True,
    )

    mutable_results = mutable_shard.query(request)
    immutable_results = immutable_shard.query(request)

    # Merge and deduplicate by ID, keeping highest score
    seen_ids = set()
    merged = []

    all_results = sorted(
        list(mutable_results) + list(immutable_results),
        key=lambda r: r.score,
        reverse=True,
    )

    for r in all_results:
        if r.id not in seen_ids:
            seen_ids.add(r.id)
            merged.append(r)
        if len(merged) >= limit:
            break

    return merged

# After the meeting, search across all history + today's captures
queries = [
    "What did Sarah propose about the architecture?",
    "battery drain and power consumption issues",
    "product comparison demo results",
]

for q in queries:
    print(f"Unified query: '{q}'")
    results = unified_query(q)
    for r in results:
        source = r.payload.get("source", "glasses")
        week = r.payload.get("week", "today")
        label = f"week {week}" if isinstance(week, int) else "today"
        print(f"  [{r.score:.3f}] [{label}] {r.payload.get('text', 'N/A')}")
    print()

## 9. Cleanup

In [ ]:
mutable_shard.close()
immutable_shard.close()

import shutil
shutil.rmtree(MUTABLE_DIR, ignore_errors=True)
shutil.rmtree(IMMUTABLE_DIR, ignore_errors=True)
print("Cleaned up")

## Summary

In this lesson you built the **meeting use case** from the Qdrant Context Hub:

- **Glasses captured** 26 meeting observations: whiteboard diagrams, discussion points, and action items
- **Cloud stored** 35 historical meeting records spanning 3 weeks of team activity
- **Cascade queries** found today's action items locally and retrieved historical partnership decisions from the cloud
- **Unified queries** merged results across both shards with deduplication

Technical patterns covered:
- Compile the same model for multiple Snapdragon devices via AI Hub
- Dual-shard architecture (mutable for local writes, immutable for server mirror)
- Dual-write pattern: local storage + sync queue
- Cascade queries: local first, cloud fallback
- Edge-to-cloud sync and snapshot-based device initialization

In the next lesson, you'll apply everything to build a phone photo memory app you can search with plain English.